# TELEX 

In [ ]:
import requests
import pandas as pd

from lxml import etree
from io import BytesIO
from collections import Counter
from datetime import datetime, timedelta


# =====================================================
# SETTINGS
# =====================================================

HEADERS = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )

}


# how many days to inspect

MAX_DAYS = 30


# =====================================================
# GENERATE DAILY SITEMAP URL
# =====================================================

def make_sitemap_url(date_obj):

    return (

        f"https://telex.hu/sitemap/"
        f"{date_obj.year}/"
        f"{date_obj.month:02d}/"
        f"{date_obj.day:02d}/"
        f"news.xml"

    )


# =====================================================
# GET URLS FROM SITEMAP
# =====================================================

def get_urls_from_sitemap(sitemap_url):

    try:

        response = requests.get(

            sitemap_url,

            headers=HEADERS,

            timeout=30

        )


        if response.status_code != 200:

            return []


        tree = etree.parse(

            BytesIO(response.content)

        )


        namespaces = {

            "sm": "https://www.sitemaps.org/schemas/sitemap/0.9"

        }


        urls = tree.xpath(

            "//sm:url/sm:loc/text()",

            namespaces=namespaces

        )


        return urls


    except Exception as e:

        print(f"\nERROR sitemap\n{sitemap_url}")

        print(e)

        return []


# =====================================================
# MAIN
# =====================================================

all_urls = []


start_date = datetime(
    2024,
    1,
    1
)


current_date = start_date


for i in range(MAX_DAYS):

    sitemap_url = make_sitemap_url(
        current_date
    )


    print(f"\n{sitemap_url}")


    urls = get_urls_from_sitemap(
        sitemap_url
    )


    print(f"URLs: {len(urls)}")


    all_urls.extend(urls)


    current_date += timedelta(days=1)


# =====================================================
# EXTRACT CATEGORIES
# =====================================================

categories = []


for url in all_urls:

    try:

        path = url.replace(
            "https://telex.hu/",
            ""
        )


        parts = [

            p for p in path.split("/")
            if p

        ]


        if len(parts) == 0:

            continue


        category = parts[0]

        categories.append(
            category
        )

    except:
        pass


# =====================================================
# CATEGORY COUNTS
# =====================================================

counter = Counter(
    categories
)


category_df = pd.DataFrame(

    counter.items(),

    columns=[

        "category",
        "count"

    ]

)


category_df = category_df.sort_values(

    "count",

    ascending=False

)


# =====================================================
# RESULTS
# =====================================================

print("\nTOTAL URLS\n")

print(len(all_urls))


print("\nUNIQUE CATEGORIES\n")

print(len(counter))


print("\nCATEGORY DISTRIBUTION\n")

print(

    category_df
    .to_string(index=False)

)


# =====================================================
# SAMPLE URLS PER CATEGORY
# =====================================================

print("\nSAMPLE URLS\n")


top_categories = category_df.head(20)[
    "category"
].tolist()


for category in top_categories:

    print("\n========================\n")

    print(category)


    examples = []


    for url in all_urls:

        if f"https://telex.hu/{category}/" in url:

            examples.append(url)


        if len(examples) >= 5:

            break


    for ex in examples:

        print(ex)


print("\nfinished")

## metadata load 

In [ ]:
import requests
import pandas as pd
import re

from bs4 import BeautifulSoup
from lxml import etree
from io import BytesIO
from tqdm import tqdm
from datetime import datetime, timedelta


# settings

headers = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )

}


output_csv = "telex_article_dataset_full.csv"


allowed_categories = {

    "kulfold",
    "belfold",
    "gazdasag",
    "g7",
    "zacc",
    "english",
    "karakter",
    "valasztasi-foldrajz",
    "napirajz"

}


# sitemap url

def make_sitemap_url(date_obj):

    return (

        f"https://telex.hu/sitemap/"
        f"{date_obj.year}/"
        f"{date_obj.month:02d}/"
        f"{date_obj.day:02d}/"
        f"news.xml"

    )


# get urls from sitemap

def get_urls_from_sitemap(sitemap_url):

    try:

        response = requests.get(

            sitemap_url,

            headers=headers,

            timeout=30

        )


        if response.status_code != 200:

            return []


        tree = etree.parse(

            BytesIO(response.content)

        )


        namespaces = {

            "sm": "https://www.sitemaps.org/schemas/sitemap/0.9"

        }


        urls = tree.xpath(

            "//sm:url/sm:loc/text()",

            namespaces=namespaces

        )


        return urls


    except Exception as e:

        print(f"\nerror sitemap\n{sitemap_url}")

        print(e)

        return []


# filter urls

def filter_urls(urls):

    filtered = []


    for url in urls:

        try:

            path = url.replace(

                "https://telex.hu/",

                ""

            )


            parts = [

                p for p in path.split("/")
                if p

            ]


            if len(parts) == 0:

                continue


            category = parts[0]


            if category in allowed_categories:

                filtered.append(url)


        except:
            pass


    return filtered


# extract article text

def extract_article_text(soup):

    containers = []


    article_tag = soup.find("article")


    if article_tag:

        containers.append(article_tag)


    for cls in [

        "article",
        "content",
        "RichText",
        "content-wrapper",
        "post-content"

    ]:

        found = soup.find_all(

            class_=re.compile(
                cls,
                re.I
            )

        )

        containers.extend(found)


    if len(containers) == 0:

        containers = [soup]


    paragraphs = []


    for container in containers:

        ps = container.find_all("p")


        for p in ps:

            text = p.get_text(

                " ",

                strip=True

            )


            if not text:

                continue


            if len(text) < 40:

                continue


            paragraphs.append(text)


    paragraphs = list(

        dict.fromkeys(paragraphs)

    )


    article_text = "\n\n".join(
        paragraphs
    )


    return article_text


# extract article data

def extract_article_data(article_url):

    try:

        response = requests.get(

            article_url,

            headers=headers,

            timeout=30

        )


        if response.status_code != 200:

            return None


        soup = BeautifulSoup(

            response.text,

            "lxml"

        )


        # title

        title = None


        title_meta = soup.find(

            "meta",

            property="og:title"

        )


        if title_meta:

            title = title_meta.get(
                "content"
            )


        # publication date

        publication_date = None


        date_candidates = [

            soup.find(
                "meta",
                property="article:published_time"
            ),

            soup.find(
                "meta",
                attrs={"name": "pubdate"}
            ),

            soup.find(
                "meta",
                property="og:published_time"
            ),

            soup.find(
                "time"
            )

        ]


        for candidate in date_candidates:

            if not candidate:
                continue


            content = (

                candidate.get("content")

                or

                candidate.get("datetime")

                or

                candidate.get_text(strip=True)

            )


            if content:

                publication_date = content

                break


        # category

        category = None


        path = article_url.replace(

            "https://telex.hu/",

            ""

        )


        parts = path.split("/")


        if len(parts) > 0:

            category = parts[0]


        # text

        article_text = extract_article_text(
            soup
        )


        return {

            "article_url": article_url,

            "title": title,

            "publication_date": publication_date,

            "category": category,

            "article_text": article_text

        }


    except Exception as e:

        print(f"\nerror article\n{article_url}")

        print(e)

        return None


# main

all_data = []


for year in range(2022, 2027):

    print(f"\n========== year {year} ==========")


    current_date = datetime(
        year,
        1,
        1
    )


    end_date = datetime(
        year,
        12,
        31
    )


    while current_date <= end_date:


        sitemap_url = make_sitemap_url(
            current_date
        )


        print(f"\n{sitemap_url}")


        urls = get_urls_from_sitemap(
            sitemap_url
        )


        if len(urls) == 0:

            current_date += timedelta(days=1)

            continue


        print(f"total urls: {len(urls)}")


        filtered_urls = filter_urls(
            urls
        )


        print(

            f"relevant urls: "
            f"{len(filtered_urls)}"

        )


        for article_url in tqdm(filtered_urls):


            article_data = extract_article_data(

                article_url

            )


            if article_data:

                all_data.append(
                    article_data
                )


        current_date += timedelta(days=1)


# dataframe

df = pd.DataFrame(
    all_data
)


# remove duplicate urls

df = df.drop_duplicates(

    subset=["article_url"]

)


# diagnostics

print("\nrows\n")

print(len(df))


print("\ncategory distribution\n")

print(

    df["category"]
    .value_counts()

)


df["text_length"] = (

    df["article_text"]
    .fillna("")
    .str.len()

)


print("\ntext length stats\n")

print(

    df["text_length"]
    .describe()

)


print("\nmissing texts\n")

print(

    df["article_text"]
    .isna()
    .sum()

)


print("\nmissing dates\n")

print(

    df["publication_date"]
    .isna()
    .sum()

)


# random sample

print("\nrandom sample\n")


sample = df.sample(
    min(5, len(df))
)


for idx, row in sample.iterrows():

    print("\n========================\n")

    print("url\n")

    print(row["article_url"])


    print("\ncategory\n")

    print(row["category"])


    print("\ntitle\n")

    print(row["title"])


    print("\ntext preview\n")

    print(

        row["article_text"][:1000]

    )


# save

df.to_csv(

    output_csv,

    index=False,

    encoding="utf-8-sig"

)


print("\nsaved\n")

print(output_csv)


print("\nfinished")

In [ ]:
import pandas as pd


# settings

file = "telex_article_dataset_full.csv"


# load

df = pd.read_csv(file)


# text length

df["text_length"] = (

    df["article_text"]
    .fillna("")
    .str.len()

)


# rows

print("\nrows\n")

print(len(df))


# duplicates

print("\nduplicate urls\n")

print(

    df["article_url"]
    .duplicated()
    .sum()

)


# missing values

print("\nmissing values\n")

print(

    df.isna()
    .sum()

)


# category distribution

print("\ncategory distribution\n")

print(

    df["category"]
    .value_counts()

)


# text length stats

print("\ntext length stats\n")

print(

    df["text_length"]
    .describe()

)


# empty texts

empty = df[

    df["text_length"] == 0

]

print("\nempty texts\n")

print(len(empty))


# short texts

short = df[

    df["text_length"] < 300

]

print("\ntexts < 300 chars\n")

print(len(short))


# long texts

long = df[

    df["text_length"] > 30000

]

print("\ntexts > 30000 chars\n")

print(len(long))


# years

print("\nyear sample\n")

if "publication_date" in df.columns:

    print(

        df["publication_date"]
        .dropna()
        .astype(str)
        .head(20)

    )


# random sample

print("\nrandom sample\n")

sample = df.sample(
    min(10, len(df))
)

for _, row in sample.iterrows():

    print("\n====================\n")

    print(row["article_url"])

    print("\ncategory\n")

    print(row["category"])

    print("\nlength\n")

    print(row["text_length"])

    print("\ntitle\n")

    print(row["title"])

    print("\npreview\n")

    print(row["article_text"][:700])


# shortest articles

print("\n10 shortest articles\n")

shortest = df.sort_values(
    "text_length"
).head(10)

print(

    shortest[
        [
            "category",
            "text_length",
            "article_url"
        ]
    ]

)


# longest articles

print("\n10 longest articles\n")

longest = df.sort_values(
    "text_length",
    ascending=False
).head(10)

print(

    longest[
        [
            "category",
            "text_length",
            "article_url"
        ]
    ]

)

In [ ]:
import pandas as pd
import re


# settings

input_file = "telex_article_dataset_full.csv"

output_file = "telex_ukraine_russia_high_recall.csv"


# load corpus

df = pd.read_csv(input_file)


print("\ninput corpus\n")

print(len(df))


# anchor keywords

anchor_patterns = [

    # ukraine

    r"ukrajn\w*",
    r"ukrán\w*",

    # russia

    r"orosz\w*",
    r"oroszország\w*",

    # putin

    r"putyin\w*",

    # zelensky

    r"zelenszk\w*",
    r"zelensky\w*",

    # kremlin

    r"kreml\w*",

    # russian officials

    r"lavrov\w*",
    r"peszkov\w*",
    r"prigozsin\w*",
    r"medvegyev\w*",

    # ukrainian officials

    r"kuleba\w*",
    r"budanov\w*"

]


# war / geopolitical context

context_patterns = [

    # war

    r"háború\w*",
    r"invázió\w*",
    r"offenzív\w*",
    r"támadás\w*",
    r"ostrom\w*",

    # military

    r"katona\w*",
    r"hadsereg\w*",
    r"fegyver\w*",
    r"rakéta\w*",
    r"drón\w*",
    r"tank\w*",
    r"légvédelem\w*",
    r"bombáz\w*",

    # nato

    r"nato\w*",

    # occupied territories

    r"donbasz\w*",
    r"doneck\w*",
    r"donyeck\w*",
    r"luhanszk\w*",
    r"krím\w*",

    # cities

    r"kijev\w*",
    r"kyiv\w*",
    r"harkiv\w*",
    r"odessza\w*",
    r"mariupol\w*",
    r"herszon\w*",
    r"zaporizzsja\w*",

    # moscow

    r"moszkva\w*",

    # sanctions

    r"szankció\w*",

    # refugees

    r"menekült\w*",

    # occupation

    r"megszáll\w*",

    # mobilization

    r"mozgósítás\w*",

    # energy

    r"energia\w*",
    r"energiaválság\w*",
    r"gázszállítás\w*",
    r"északi\s*áramlat\w*",
    r"nord\s*stream\w*"

]


# compile regex

anchor_regex = re.compile(

    "|".join(anchor_patterns),

    flags=re.IGNORECASE

)


context_regex = re.compile(

    "|".join(context_patterns),

    flags=re.IGNORECASE

)


# combine text

df["combined_text"] = (

    df["title"]
    .fillna("")
    + " "
    + df["article_text"]
    .fillna("")

)


# filtering

filtered = df[

    df["combined_text"]
    .str.contains(
        anchor_regex,
        na=False
    )

    &

    df["combined_text"]
    .str.contains(
        context_regex,
        na=False
    )

]


# deduplicate

filtered = filtered.drop_duplicates(

    subset=["article_url"]

)


# diagnostics

print("\nfiltered corpus\n")

print(len(filtered))


print("\ncategory distribution\n")

print(

    filtered["category"]
    .value_counts()

)


# text length

filtered["text_length"] = (

    filtered["article_text"]
    .fillna("")
    .str.len()

)


print("\ntext length stats\n")

print(

    filtered["text_length"]
    .describe()

)


print("\narticles with text\n")

print(

    (filtered["text_length"] > 0)
    .value_counts()

)


# random sample

print("\nrandom sample\n")


sample = filtered.sample(

    min(10, len(filtered)),

    random_state=42

)


for idx, row in sample.iterrows():

    print("\n========================\n")

    print(row["article_url"])

    print("\ncategory\n")

    print(row["category"])

    print("\ntitle\n")

    print(row["title"])

    print("\ntext preview\n")

    print(

        row["article_text"][:1000]

    )


# save

filtered.to_csv(

    output_file,

    index=False,

    encoding="utf-8-sig"

)


print("\nfiltered corpus saved\n")

print(output_file)


print("\nfinished")

### Date and image url 

In [ ]:
import pandas as pd
import requests

from bs4 import BeautifulSoup


# load

df = pd.read_csv(
    "telex_article_dataset_full.csv"
)


sample = df.sample(
    10,
    random_state=42
)


headers = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )

}


for url in sample["article_url"]:

    print("\n====================\n")

    print(url)

    try:

        response = requests.get(
            url,
            headers=headers,
            timeout=30
        )

        soup = BeautifulSoup(
            response.text,
            "lxml"
        )

        metas = soup.find_all("meta")

        for meta in metas:

            prop = meta.get("property")
            name = meta.get("name")

            if prop and "date" in prop.lower():

                print(
                    "property:",
                    prop,
                    meta.get("content")
                )

            if name and "date" in name.lower():

                print(
                    "name:",
                    name,
                    meta.get("content")
                )

    except Exception as e:

        print(e)

In [ ]:
import pandas as pd
import re


file = "telex_article_dataset_full.csv"


df = pd.read_csv(file)


def extract_date(url):

    match = re.search(

        r"/(\d{4})/(\d{2})/(\d{2})/",

        str(url)

    )

    if match:

        year, month, day = match.groups()

        return f"{year}-{month}-{day}"

    return None


df["publication_date"] = (

    df["article_url"]

    .apply(extract_date)

)


print("\nmissing dates\n")

print(

    df["publication_date"]

    .isna()

    .sum()

)


print("\nyear distribution\n")

print(

    pd.to_datetime(

        df["publication_date"]

    )

    .dt.year

    .value_counts()

    .sort_index()

)


df.to_csv(

    file,

    index=False,

    encoding="utf-8-sig"

)


print("\ndates added\n")

In [ ]:
import pandas as pd
import requests

from bs4 import BeautifulSoup


# settings

input_file = "telex_ukraine_russia_high_recall.csv"

sample_size = 10


# load

df = pd.read_csv(input_file)

sample = df.sample(
    sample_size,
    random_state=42
)


headers = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )

}


def extract_article_images(url):

    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    soup = BeautifulSoup(
        response.text,
        "lxml"
    )

    article = soup.find("article")

    if article is None:
        return []

    images = []

    for img in article.find_all("img"):

        src = (

            img.get("src")
            or img.get("data-src")
            or ""

        )

        if src:

            images.append(src)

    # remove duplicates

    images = list(
        dict.fromkeys(images)
    )

    return images


for _, row in sample.iterrows():

    url = row["article_url"]

    print("\n========================\n")

    print("article")

    print(url)

    try:

        images = extract_article_images(
            url
        )

        print(
            f"\nimages found: {len(images)}\n"
        )

        for img in images:

            print(img)

    except Exception as e:

        print(e)

In [ ]:
import pandas as pd
import requests

from bs4 import BeautifulSoup


df = pd.read_csv(
    "telex_ukraine_russia_high_recall.csv"
)

sample = df.sample(
    10,
    random_state=42
)


for url in sample["article_url"]:

    print("\n====================\n")
    print(url)

    try:

        response = requests.get(
            url,
            timeout=30
        )

        soup = BeautifulSoup(
            response.text,
            "lxml"
        )

        tag = soup.find(
            "meta",
            property="og:image"
        )

        if tag:

            print(
                tag.get("content")
            )

        else:

            print("NO OG IMAGE")

    except Exception as e:

        print(e)

In [ ]:
import pandas as pd
import requests

from bs4 import BeautifulSoup


input_file = "telex_ukraine_russia_high_recall.csv"

df = pd.read_csv(input_file)

sample = df.sample(
    20,
    random_state=42
)


headers = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )

}


for _, row in sample.iterrows():

    url = row["article_url"]

    print("\n========================\n")

    print(url)

    try:

        response = requests.get(
            url,
            headers=headers,
            timeout=30
        )

        soup = BeautifulSoup(
            response.text,
            "lxml"
        )

        article = soup.find("article")

        if article is None:

            print("NO ARTICLE TAG")

            continue

        images = []

        for img in article.find_all("img"):

            src = (

                img.get("src")
                or img.get("data-src")
                or ""
            )

            if src:

                images.append(src)

        images = list(
            dict.fromkeys(images)
        )

        print(
            f"images found: {len(images)}"
        )

        for img in images:

            print(img)

    except Exception as e:

        print(e)

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://telex.hu/kulfold/2022/05/14/orosz-ukran-haboru-80-nap"

headers = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

soup = BeautifulSoup(
    response.text,
    "lxml"
)

print("\nall images\n")

for i, img in enumerate(soup.find_all("img"), 1):

    src = (
        img.get("src")
        or img.get("data-src")
        or ""
    )

    if src:

        print(f"\n{i}")
        print(src)

### Image url download

In [ ]:
import pandas as pd
import requests

from bs4 import BeautifulSoup
from tqdm import tqdm


# settings

input_file = "telex_ukraine_russia_high_recall.csv"

output_file = "telex_ukraine_russia_with_images.csv"


# load

df = pd.read_csv(input_file)

print("\nrows\n")

print(len(df))


# headers

headers = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )

}


# extractor

def extract_images(url):

    try:

        response = requests.get(
            url,
            headers=headers,
            timeout=30
        )

        if response.status_code != 200:

            return None, []

        soup = BeautifulSoup(
            response.text,
            "lxml"
        )

        # og image

        og_image = None

        tag = soup.find(
            "meta",
            property="og:image"
        )

        if tag:

            og_image = tag.get(
                "content"
            )

        # body images

        body_images = []

        for img in soup.find_all("img"):

            src = img.get("src")

            if not src:
                continue

            src = str(src)

            if (
                "hero-full" in src
                or "cikktorzs" in src
            ):

                body_images.append(src)

        # deduplicate

        body_images = list(
            dict.fromkeys(body_images)
        )

        return og_image, body_images

    except:

        return None, []


# process

og_images = []

body_images_all = []

body_image_counts = []

has_body_images = []


for url in tqdm(df["article_url"]):

    og_image, body_images = extract_images(
        url
    )

    og_images.append(
        og_image
    )

    body_images_all.append(
        str(body_images)
    )

    body_image_counts.append(
        len(body_images)
    )

    has_body_images.append(
        len(body_images) > 0
    )


# save columns

df["og_image"] = og_images

df["body_images"] = body_images_all

df["body_image_count"] = body_image_counts

df["has_body_images"] = has_body_images


# diagnostics

print("\nog images found\n")

print(

    df["og_image"]
    .notna()
    .sum()

)

print("\nmissing og images\n")

print(

    df["og_image"]
    .isna()
    .sum()

)

print("\narticles with body images\n")

print(

    df["has_body_images"]
    .value_counts()

)

print("\nbody image count stats\n")

print(

    df["body_image_count"]
    .describe()

)


# save

df.to_csv(

    output_file,

    index=False,

    encoding="utf-8-sig"

)


print("\nsaved\n")

print(output_file)

print("\nfinished")

In [ ]:
import pandas as pd

df = pd.read_csv(
    "telex_ukraine_russia_with_images.csv"
)

no_images = df[

    df["og_image"].isna()

    &

    (
        df["body_image_count"] == 0
    )

]

print("\narticles without any images\n")

print(len(no_images))

In [ ]:
import pandas as pd
import ast


# load

df = pd.read_csv(
    "telex_ukraine_russia_with_images.csv"
)


# collect all images

all_images = []


# og images

og_images = (

    df["og_image"]
    .dropna()
    .tolist()

)

all_images.extend(
    og_images
)


# body images

for imgs in df["body_images"]:

    try:

        parsed = ast.literal_eval(imgs)

        all_images.extend(parsed)

    except:

        pass


# diagnostics

print("\nall image urls\n")

print(len(all_images))


# unique

unique_images = list(
    set(all_images)
)

print("\nunique image urls\n")

print(len(unique_images))


# duplicates

print("\nduplicate image urls\n")

print(

    len(all_images)
    - len(unique_images)

)


# most repeated

image_counts = pd.Series(
    all_images
).value_counts()


print("\nmost repeated images\n")

print(

    image_counts.head(30)

)

In [ ]:
import pandas as pd
import ast


# load

df = pd.read_csv(
    "telex_ukraine_russia_with_images.csv"
)


# collect all images

all_images = []


# og images

all_images.extend(

    df["og_image"]
    .dropna()
    .tolist()

)


# body images

for imgs in df["body_images"]:

    try:

        parsed = ast.literal_eval(imgs)

        all_images.extend(parsed)

    except:

        pass


# count

image_counts = pd.Series(
    all_images
).value_counts()


# repeated > 5

repeated = image_counts[
    image_counts > 3
]


print("\nimages repeated > 5 times\n")

print(len(repeated))


print("\nmost repeated\n")

print(
    repeated.head(100)
)

In [ ]:
import pandas as pd
import ast


# settings

input_file = "telex_ukraine_russia_with_images.csv"

output_file = "telex_ukraine_russia_with_images_cleaned.csv"


# load

df = pd.read_csv(input_file)


# bad images

bad_images = {

    "https://telex.hu/telex_logo_fb_share.png",

    "https://telex.hu/g7_og_image_fb.jpg",

    "null/null",

    "https://telex.hu/after_og_image_fb.jpg",

    "https://telex.hu/karakter_facebook_og_ajanlo_kep.jpg"

}


# clean og_image

df["og_image"] = df["og_image"].apply(

    lambda x:

    None

    if str(x) in bad_images

    else x

)


# clean body_images

cleaned_body_images = []

for imgs in df["body_images"]:

    try:

        parsed = ast.literal_eval(imgs)

    except:

        parsed = []


    cleaned = [

        img

        for img in parsed

        if img not in bad_images

    ]


    cleaned = list(
        dict.fromkeys(cleaned)
    )

    cleaned_body_images.append(
        cleaned
    )


# save cleaned lists

df["body_images"] = cleaned_body_images


# recompute diagnostics

df["body_image_count"] = (

    df["body_images"]

    .apply(len)

)


df["has_body_images"] = (

    df["body_image_count"] > 0

)


# articles without any images

df["has_any_image"] = (

    df["og_image"].notna()

    |

    df["has_body_images"]

)


# diagnostics

print("\narticles with any image\n")

print(

    df["has_any_image"]
    .value_counts()

)


print("\narticles without any image\n")

print(

    (~df["has_any_image"])
    .sum()

)


print("\nbody image stats\n")

print(

    df["body_image_count"]
    .describe()

)


# save

df.to_csv(

    output_file,

    index=False,

    encoding="utf-8-sig"

)


print("\nsaved\n")

print(output_file)

print("\nfinished")

In [ ]:
import pandas as pd
import ast
import requests
import os
import hashlib

from tqdm import tqdm


# settings

input_file = "telex_ukraine_russia_with_images_cleaned.csv"

output_dir = "telex_ukraine_russia_images"

failed_file = "telex_failed_downloads.csv"

timeout = 30


# create folder

os.makedirs(
    output_dir,
    exist_ok=True
)


# load dataset

df = pd.read_csv(input_file)

print("\nrows\n")

print(len(df))


# collect all image urls

all_images = []


# og images

all_images.extend(

    df["og_image"]
    .dropna()
    .tolist()

)


# body images

for imgs in df["body_images"]:

    try:

        parsed = ast.literal_eval(imgs)

        all_images.extend(parsed)

    except:

        pass


# unique only

unique_images = sorted(

    list(set(all_images))

)


print("\nunique images\n")

print(len(unique_images))


# headers

headers = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )

}


# filename helper

def make_filename(url):

    url_hash = hashlib.md5(

        url.encode()

    ).hexdigest()


    lower = url.lower()


    if ".jpg" in lower:

        ext = ".jpg"

    elif ".jpeg" in lower:

        ext = ".jpeg"

    elif ".png" in lower:

        ext = ".png"

    elif ".webp" in lower:

        ext = ".webp"

    elif ".avif" in lower:

        ext = ".avif"

    else:

        ext = ".jpg"


    return url_hash + ext


# download

failed = []

downloaded = 0


for url in tqdm(unique_images):

    try:

        filename = make_filename(
            url
        )

        filepath = os.path.join(
            output_dir,
            filename
        )


        # skip existing

        if os.path.exists(filepath):

            downloaded += 1

            continue


        response = requests.get(

            url,

            headers=headers,

            timeout=timeout,

            stream=True

        )


        if response.status_code != 200:

            failed.append({

                "image_url": url,

                "status_code": response.status_code

            })

            continue


        with open(
            filepath,
            "wb"
        ) as f:

            for chunk in response.iter_content(
                8192
            ):

                f.write(chunk)


        downloaded += 1


    except Exception as e:

        failed.append({

            "image_url": url,

            "error": str(e)

        })


# failed log

failed_df = pd.DataFrame(
    failed
)


failed_df.to_csv(

    failed_file,

    index=False,

    encoding="utf-8-sig"

)


# diagnostics

print("\ndownloaded\n")

print(downloaded)


print("\nfailed\n")

print(len(failed))


print("\noutput dir\n")

print(output_dir)


print("\nfailed log\n")

print(failed_file)


print("\nfinished")

In [ ]:
import os
import time
import hashlib
import pandas as pd
import requests

failed_file = "telex_failed_downloads.csv"
output_dir = "telex_ukraine_russia_images"

os.makedirs(output_dir, exist_ok=True)

failed = pd.read_csv(failed_file)

urls = (
    failed["image_url"]
    .dropna()
    .astype(str)
    .unique()
)

print()
print("urls to retry")
print(len(urls))

new_failed = []

downloaded = 0
skipped = 0

session = requests.Session()

headers = {
    "User-Agent": "Mozilla/5.0"
}

for i, url in enumerate(urls):

    if i % 10 == 0:
        print(
            f"{i}/{len(urls)} "
            f"downloaded={downloaded} "
            f"skipped={skipped} "
            f"failed={len(new_failed)}"
        )

    try:

        ext = os.path.splitext(
            url.split("?")[0]
        )[1]

        if ext.lower() not in [
            ".jpg",
            ".jpeg",
            ".png",
            ".webp",
            ".gif"
        ]:
            ext = ".jpg"

        filename = (
            hashlib.md5(
                url.encode("utf-8")
            ).hexdigest()
            + ext
        )

        filepath = os.path.join(
            output_dir,
            filename
        )

        if os.path.exists(filepath):
            skipped += 1
            continue

        r = session.get(
            url,
            headers=headers,
            timeout=20
        )

        if r.status_code != 200:

            new_failed.append({
                "image_url": url,
                "status_code": r.status_code
            })

            continue

        with open(filepath, "wb") as f:
            f.write(r.content)

        downloaded += 1

        time.sleep(0.2)

    except Exception as e:

        new_failed.append({
            "image_url": url,
            "error": str(e)
        })

print()
print("downloaded")
print(downloaded)

print()
print("skipped")
print(skipped)

print()
print("failed")
print(len(new_failed))

pd.DataFrame(
    new_failed
).to_csv(
    "telex_failed_downloads_retry.csv",
    index=False,
    encoding="utf-8-sig"
)

print()
print("saved")
print("telex_failed_downloads_retry.csv")

In [ ]:
import pandas as pd
import os
import ast
import hashlib

df = pd.read_csv(
    "telex_ukraine_russia_with_images_cleaned.csv"
)

image_dir = "telex_ukraine_russia_images"

all_urls = set()

# og images

for url in df["og_image"].dropna():

    url = str(url).strip()

    if url and url != "nan":
        all_urls.add(url)

# body images

for images in df["body_images"].dropna():

    try:

        img_list = ast.literal_eval(images)

        for url in img_list:

            if isinstance(url, str):

                url = url.strip()

                if url:
                    all_urls.add(url)

    except:
        pass

print()
print("unique image urls expected")
print(len(all_urls))

missing_urls = []

for url in all_urls:

    ext = os.path.splitext(
        url.split("?")[0]
    )[1]

    if ext.lower() not in [
        ".jpg",
        ".jpeg",
        ".png",
        ".webp",
        ".gif"
    ]:
        ext = ".jpg"

    filename = (
        hashlib.md5(
            url.encode("utf-8")
        ).hexdigest()
        + ext
    )

    filepath = os.path.join(
        image_dir,
        filename
    )

    if not os.path.exists(filepath):

        missing_urls.append(url)

print()
print("images already on disk")
print(len(all_urls) - len(missing_urls))

print()
print("images missing")
print(len(missing_urls))

pd.DataFrame({
    "image_url": missing_urls
}).to_csv(
    "telex_missing_images.csv",
    index=False,
    encoding="utf-8-sig"
)

print()
print("saved")
print("telex_missing_images.csv")

In [ ]:
import pandas as pd
import requests
import os
import hashlib
import time
import random

# ------------------
# settings
# ------------------

INPUT_FILE = "telex_missing_images.csv"

OUTPUT_DIR = "telex_ukraine_russia_images"

SLEEP_RANGE = (0.5, 1.0)

TIMEOUT = (10, 60)

# ------------------
# load urls
# ------------------

df = pd.read_csv(INPUT_FILE)

urls = (
    df["image_url"]
    .dropna()
    .astype(str)
    .unique()
)

print()
print("urls to download")
print(len(urls))

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

# ------------------
# session
# ------------------

session = requests.Session()

headers = {
    "User-Agent":
    (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 "
        "Safari/537.36"
    )
}

# ------------------
# stats
# ------------------

downloaded = 0
skipped = 0
failed = []

# ------------------
# download
# ------------------

for i, url in enumerate(urls):

    if i % 100 == 0:

        print(
            f"{i}/{len(urls)} "
            f"downloaded={downloaded} "
            f"skipped={skipped} "
            f"failed={len(failed)}"
        )

    try:

        ext = os.path.splitext(
            url.split("?")[0]
        )[1].lower()

        if ext not in [
            ".jpg",
            ".jpeg",
            ".png",
            ".webp",
            ".gif"
        ]:
            ext = ".jpg"

        filename = (
            hashlib.md5(
                url.encode("utf-8")
            ).hexdigest()
            + ext
        )

        filepath = os.path.join(
            OUTPUT_DIR,
            filename
        )

        if os.path.exists(filepath):

            skipped += 1
            continue

        response = session.get(
            url,
            headers=headers,
            timeout=TIMEOUT,
            stream=True
        )

        if response.status_code != 200:

            failed.append({
                "image_url": url,
                "status_code": response.status_code
            })

            continue

        with open(filepath, "wb") as f:

            for chunk in response.iter_content(
                chunk_size=8192
            ):

                if chunk:
                    f.write(chunk)

        downloaded += 1

        time.sleep(
            random.uniform(
                *SLEEP_RANGE
            )
        )

    except Exception as e:

        failed.append({
            "image_url": url,
            "error": str(e)
        })

        time.sleep(3)

# ------------------
# save failed
# ------------------

failed_df = pd.DataFrame(
    failed
)

failed_df.to_csv(
    "telex_missing_images_failed.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------
# summary
# ------------------

print()
print("downloaded")
print(downloaded)

print()
print("skipped")
print(skipped)

print()
print("failed")
print(len(failed))

print()
print("failed log")
print("telex_missing_images_failed.csv")

print()
print("finished")

In [ ]:
import pandas as pd
import os
import ast
import hashlib

df = pd.read_csv(
    "telex_ukraine_russia_with_images_cleaned.csv"
)

all_urls = set()

for url in df["og_image"].dropna():

    if str(url) != "nan":
        all_urls.add(str(url))

for imgs in df["body_images"].dropna():

    try:

        for url in ast.literal_eval(imgs):

            all_urls.add(url)

    except:
        pass

missing = []

for url in all_urls:

    ext = os.path.splitext(
        url.split("?")[0]
    )[1]

    if ext.lower() not in [
        ".jpg",
        ".jpeg",
        ".png",
        ".webp",
        ".gif"
    ]:
        ext = ".jpg"

    filename = (
        hashlib.md5(
            url.encode("utf-8")
        ).hexdigest()
        + ext
    )

    filepath = os.path.join(
        "telex_ukraine_russia_images",
        filename
    )

    if not os.path.exists(filepath):

        missing.append(url)

print("still missing")
print(len(missing))